In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression

data1 = pd.read_csv('homework_8.1.csv')

In [16]:
data1.head()

,X,Y,Z
0,1,4.109218,1.764052
1,0,2.259504,0.400157
2,0,-0.647584,0.978738
3,0,2.106071,2.240893
4,1,3.583464,1.867558


In [17]:
X = data1[['Z']]

y = data1['X']

model_lr = LogisticRegression()

model_lr.fit(X, y)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [18]:
data1['propensity_score'] = model_lr.predict_proba(X)[:, 1]

data1.head()

,X,Y,Z,propensity_score
0,1,4.109218,1.764052,0.840114
1,0,2.259504,0.400157,0.584646
2,0,-0.647584,0.978738,0.711082
3,0,2.106071,2.240893,0.892793
4,1,3.583464,1.867558,0.853089


In [22]:
weighted_y_treated = []
weighted_y_untreated = []

for i in range(len(data1)):
    if data1.loc[i, 'X'] == 1:
        ipw = 1/data1.loc[i, 'propensity_score']
        weighted_y_treated.append(ipw * data1.loc[i, 'Y'])
    else: 
        ipw = 1/(1 - data1.loc[i, 'propensity_score'])
        weighted_y_untreated.append(ipw * data1.loc[i, 'Y'])

ATE = (sum(weighted_y_treated) - sum(weighted_y_untreated))/len(data1)

print(f'The average treatment effect is {ATE:.1f}')


The average treatment effect is 2.2


In [24]:
data2 = pd.read_csv('homework_8.2.csv')

data2.head()

,X,Y,Z1,Z2
0,1,4.652085,1.764052,2.320015
1,1,2.590221,0.400157,1.292631
2,1,3.898981,0.978738,0.556423
3,1,5.857179,2.240893,2.345607
4,1,3.647489,1.867558,2.095611


In [38]:
import scipy

Z_matrix = np.array(data2[['Z1', 'Z2']]).T

cov_matrix = np.cov(Z_matrix)
inv_cov = np.linalg.inv(cov_matrix)

print('Inverted Covariance Matrix')
print(inv_cov)

Inverted Covariance Matrix
[[ 2.02734407 -1.03387633]
 [-1.03387633  1.06684813]]


In [ ]:
untreated_rows = data2[data2['X'] == 0]
treated_rows = data2[data2['X'] == 1]

matches = []

for index, treated in treated_rows.iterrows():

    distances = []

    for idx, untreated in untreated_rows.iterrows():
        distance = scipy.spatial.distance.mahalanobis(treated[['Z1', 'Z2']].to_numpy(dtype = float), untreated[['Z1', 'Z2']].to_numpy(dtype = float),
                                                      inv_cov)
        distances.append(distance)

    nearest_index = np.argmin(distances)

    matches.append((index, untreated_rows.index[nearest_index]))

matches

[(0, np.int64(681)),
 (1, np.int64(752)),
 (2, np.int64(892)),
 (3, np.int64(922)),
 (4, np.int64(922)),
 (5, np.int64(371)),
 (6, np.int64(681)),
 (9, np.int64(381)),
 (10, np.int64(704)),
 (11, np.int64(161)),
 (12, np.int64(764)),
 (13, np.int64(802)),
 (15, np.int64(51)),
 (16, np.int64(922)),
 (19, np.int64(904)),
 (21, np.int64(436)),
 (24, np.int64(161)),
 (26, np.int64(771)),
 (28, np.int64(161)),
 (29, np.int64(123)),
 (30, np.int64(991)),
 (31, np.int64(381)),
 (36, np.int64(436)),
 (37, np.int64(681)),
 (38, np.int64(986)),
 (43, np.int64(161)),
 (53, np.int64(121)),
 (54, np.int64(240)),
 (55, np.int64(381)),
 (57, np.int64(35)),
 (62, np.int64(799)),
 (64, np.int64(698)),
 (66, np.int64(506)),
 (67, np.int64(500)),
 (69, np.int64(226)),
 (72, np.int64(677)),
 (74, np.int64(500)),
 (75, np.int64(124)),
 (79, np.int64(533)),
 (81, np.int64(892)),
 (82, np.int64(217)),
 (84, np.int64(123)),
 (87, np.int64(49)),
 (88, np.int64(461)),
 (89, np.int64(677)),
 (91, np.int64(579)),